In [5]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

MODEL_PATH = "Finetuned Bert Model/checkpoint-60000"

model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

model.eval()  # สำคัญมาก


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [6]:
import re

def clean_log(text):
    # ลบ timestamp (optional แต่แนะนำ)
    text = re.sub(r"\d{4}-\d{2}-\d{2}T.*?Z", "", text)
    # แปลง tab เป็น space
    text = text.replace("\t", " ")
    return text.strip()

In [7]:
def predict_log(log_text):
    log_text = clean_log(log_text)
    inputs = tokenizer(
        log_text,
        return_tensors="pt",
        truncation=True,
        padding=True, # ใส่เผื่อเอาไว้ตอน inference มากกว่า 1 log (Batch Size > 1)
        max_length=128
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]

    return "NORMAL" if pred == 1 else "ANOMALY" ,prob

----ตรวจคำตอบ------

In [17]:
# ใช้ log บรรทัดที่ 3 (นับแถวของ column) ด้วย

predict,confidence =predict_log(
    """
2025-12-28T17:2:17.000777Z	18	Query	select balance from accounts where  account_id = 'bmoXPcst81VKz38c9';
    """
)
print(predict,confidence)

# คำตอบที่ถูกคือ normal

NORMAL [6.03062322301895e-11, 1.0]


In [ ]:
# ใช้ log บรรทัดที่ 4 (นับแถวของ column) ด้วย

print(predict_log(
    """
    "2025-12-26T23:22:26.714509Z	38	Query	select xZS-V from Ui6WRY where qujmR__YLZmt8hHqXoE = 'Aoov7ApF3xu' or "1"="1""
    """
))
# คำตอบที่ถูกคือ anomally

ANOMALY
